In [1]:
import json
import math
import re
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd

# Naive Bayes Cheese Classifier — PoC (Coarser Labelling)

**This is the PoC variant of `Naive_Bayes_Cheese_Classifier.ipynb`.**
It uses a coarser labelling scheme to produce a larger, more balanced dataset.
See the label resolution cells for the specific changes.

A direct analogy to bag-of-words spam detection:

| Spam classifier | Cheese classifier |
|-----------------|-------------------|
| spam / ham      | cheese / macro    |
| word            | entity type (unit/building) |
| word count      | peak count of that entity during the game |
| email text      | SC2 game state parquet |

## Strategy Label Schema

`strategies.json` labelling convention:

| Value | Meaning |
|-------|---------|
| `1`   | Cheese  |
| `0`   | Not cheese (macro) |
| `999` | Not a strategy — ignore |

**How to use:** Run cells top-to-bottom.

In [2]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
# This notebook lives in ML_PoC/, so the project root is one level up.
PROJECT_ROOT    = Path("..").resolve()
PARQUET_DIR     = PROJECT_ROOT / "data" / "quickstart" / "parquet"
JSON_DIR        = PROJECT_ROOT / "data" / "quickstart" / "json"
STRATEGIES_PATH = PROJECT_ROOT / "data" / "quickstart" / "strategies" / "strategies.json"

with open(STRATEGIES_PATH, "r", encoding="utf-8") as fh:
    STRATEGIES: Dict[str, Dict[str, int]] = json.load(fh)

print(f"Loaded {len(STRATEGIES)} players from strategies.json")
print(f"Parquet files found : {len(list(PARQUET_DIR.glob('*.parquet')))}")
print(f"Metadata JSON files : {len(list(JSON_DIR.glob('*_metadata.json')))}")

Loaded 51 players from strategies.json
Parquet files found : 713
Metadata JSON files : 651


## Step 1 — Label Resolution

Each game's metadata JSON records the chat messages sent during the match.
Strategy bots embed their build order as a `Tag:  <strategy_name>` message.
`strategies.json` maps each `(player_name, strategy_name)` pair to a label.

**Conflict rule:** if a player sends messages that map to *both* `0` and `1`
in the same game, the conflict is printed for manual review and that player's
entry is set to `None` (excluded from the dataset).

### PoC Coarser Labelling: Manually Curated Player-Level Fallback

The strict notebook labels games exclusively from in-game strategy tags — a
precise method, but one that leaves a large portion of games unlabelled.
Many bots either send no strategy tags at all, or send only diagnostic
messages that are discarded as noise (`999`).  The result is a dataset
heavily skewed toward cheese, because the bots that consistently emit
recognisable tags happen to be those running cheese strategies.

This PoC introduces a **player-level label fallback**.  For games where
no strategy tag can be resolved, the labeller checks two manually curated
sets — `MACRO_PLAYERS` and `CHEESE_PLAYERS` — and applies the corresponding
label if the player appears in one of them.

To tighten this player based heuristic, it has been noticed some of the 'macro' bots will perform cheesy strategies as a response to the actions of other bots. To combat this, if a known macro bot has a known cheesy strategy it will perform, I will add targetted logic to the heuristic to re-label the macro bot game as a cheese game. For example, Eris has been observed going 2-base muta, a known cheesy strategy, to punish 'greedy' or 'cheesy' opponents. So when Eris builds >5 mutas I will labell that game as cheese instead.

**Tag-based labels remain the primary source of truth.**  The player-level
fallback only activates when a game produces no resolvable tag.  This
ensures that a bot's in-game tag is never overridden by its general
tendency — important because bot behaviour can be reactive, and a bot
known for one style of play may occasionally respond to game conditions
in a way that crosses into the other class.

**The curated lists are maintained entirely by hand.**  No player name
is added programmatically.  This is a deliberate design choice: the lists
reflect specific, considered knowledge about each bot's typical behaviour,
and that knowledge must be verified and recorded by a human before any
game is labelled on that basis.  The trade-off accepted here is that
player-level labels are coarser than tag-based labels — they describe
a bot's predominant tendency, not a guarantee for every individual game.

In [3]:
# ---------------------------------------------------------------------------
# PoC — Manually curated player-level label fallbacks
# ---------------------------------------------------------------------------
# IMPORTANT: These sets must NEVER be populated programmatically.
# Only add a bot name here after manually verifying that it reliably
# plays the corresponding style. Update or remove entries as new
# information becomes available.
#
# These are used ONLY as a fallback when a game produces no resolvable
# strategy tag. Tag-based labels always take priority.
# ---------------------------------------------------------------------------

MACRO_PLAYERS: Set[str] = {"negativezero", "eris", "deimos"
    # Bot names known to play exclusively macro build orders.
    # Add entries here manually.
}

CHEESE_PLAYERS: Set[str] = {"really", "what", "why", "who",
    # Bot names known to play exclusively cheese build orders.
    # Add entries here manually.
}

print(f"Player-level fallback — macro  : {len(MACRO_PLAYERS)} bots")
print(f"Player-level fallback — cheese : {len(CHEESE_PLAYERS)} bots")

overlap = MACRO_PLAYERS & CHEESE_PLAYERS
if overlap:
    print(f"WARNING: the following bots appear in both lists — {overlap}")

Player-level fallback — macro  : 3 bots
Player-level fallback — cheese : 4 bots


In [4]:
def strip_tag_prefix(message: str) -> str:
    """
    Remove the 'Tag:  ' prefix that some bots prepend to strategy messages.

    The metadata format is: "Tag:  <strategy_name>"
    The corresponding strategies.json key is: "<strategy_name>"

    Args:
        message: Raw message string from game metadata.

    Returns:
        Strategy name with any "Tag:" prefix stripped and whitespace trimmed.

    Called by: resolve_player_label, build_game_labels
    """
    if message.startswith("Tag:"):
        return message[message.index(":") + 1:].strip()
    return message.strip()


def resolve_player_label(
    player_name: str,
    messages: List[str],
    strategies: Dict[str, Dict[str, int]],
) -> Optional[int]:
    """
    Resolve the strategy label for one player in one game via in-game tags.

    Filters out 999 (noise) labels and checks whether all remaining messages
    agree on a single label.

    Args:
        player_name: Bot name — top-level key in strategies.json.
        messages:    All raw message strings sent by this player in this game.
        strategies:  The full strategies dictionary.

    Returns:
        0 or 1 if all non-noise messages agree on a single label.
        "CONFLICT" if messages contain both a 0 and a 1 label.
        None if the player is not in strategies.json or all messages are 999/unknown.

    Called by: build_game_labels
    """
    if player_name not in strategies:
        return None

    player_strats = strategies[player_name]
    valid_labels = []

    for msg in messages:
        key = strip_tag_prefix(msg)
        if key in player_strats:
            label = player_strats[key]
            if label != 999:
                valid_labels.append(label)

    if not valid_labels:
        return None

    unique = set(valid_labels)
    if len(unique) == 1:
        return unique.pop()
    return "CONFLICT"


def build_game_labels(
    json_dir: Path,
    strategies: Dict[str, Dict[str, int]],
    macro_players: Set[str] = frozenset(),
    cheese_players: Set[str] = frozenset(),
) -> Dict[str, dict]:
    """
    Walk all metadata JSON files and produce per-player labels for every game.

    Labelling priority (per player per game):
      1. Tag-based: resolve_player_label() — uses in-game strategy messages.
      2. Player-level fallback: if no tag resolves, check macro_players /
         cheese_players and assign 0 or 1 accordingly.
      3. None: player remains unlabelled and is excluded from the dataset.

    Tag-based labels are never overridden by the player-level fallback —
    an in-game tag is always treated as the more specific signal.

    Player name matching for the fallback sets is case-insensitive — exact
    spelling is required but capitalisation does not matter.

    For each game:
      - Groups messages by player_id (1 or 2).
      - Calls resolve_player_label for each player.
      - Prints and discards conflicting tag-based labels (for manual review).
      - Applies player-level fallback for any player whose tag did not resolve.
      - Warns when neither player has a label after both steps.

    Args:
        json_dir:       Directory containing *_metadata.json files.
        strategies:     Loaded strategies.json dictionary.
        macro_players:  Manually curated set of bot names to label as macro (0)
                        when no tag resolves. Case-insensitive.
        cheese_players: Manually curated set of bot names to label as cheese (1)
                        when no tag resolves. Case-insensitive.

    Returns:
        dict keyed by match_id, e.g.:
        {
            "match_4184393": {
                "p1": {"name": "zig-reapers", "label": None},
                "p2": {"name": "really",      "label": 1},
            },
            ...
        }

    Called by: top-level dataset construction code.
    """
    # Normalise player sets to lowercase once so lookups are case-insensitive.
    # Player names from match metadata are also lowercased at lookup time.
    macro_lower  = {name.lower() for name in macro_players}
    cheese_lower = {name.lower() for name in cheese_players}

    game_labels: Dict[str, dict] = {}

    for json_path in sorted(json_dir.glob("*_metadata.json")):
        with open(json_path, encoding="utf-8") as fh:
            meta = json.load(fh)

        match_id = json_path.stem.replace("_metadata", "")
        p1_name  = meta["players"]["p1"]["name"]
        p2_name  = meta["players"]["p2"]["name"]

        # Group raw message strings by player_id (1-indexed)
        msgs: Dict[int, List[str]] = {1: [], 2: []}
        for msg_obj in meta.get("messages", {}).get("messages", []):
            msgs[msg_obj["player_id"]].append(msg_obj["message"])

        # --- Step 1: tag-based labelling (primary) ---
        p1_label = resolve_player_label(p1_name, msgs[1], strategies)
        p2_label = resolve_player_label(p2_name, msgs[2], strategies)

        # --- Conflict handling ---
        def _conflict_msgs(player_name, player_msgs, want_label):
            """Return raw messages from a player that resolved to want_label."""
            strats = strategies.get(player_name, {})
            return [m for m in player_msgs
                    if strats.get(strip_tag_prefix(m)) == want_label]

        if p1_label == "CONFLICT":
            ones  = _conflict_msgs(p1_name, msgs[1], 1)
            zeros = _conflict_msgs(p1_name, msgs[1], 0)
            print(f"CONFLICT  (p1 — {p1_name})")
            print(f"  1: {ones}")
            print(f"  0: {zeros}")
            print(f"  match: {json_path.name}\n")
            p1_label = None

        if p2_label == "CONFLICT":
            ones  = _conflict_msgs(p2_name, msgs[2], 1)
            zeros = _conflict_msgs(p2_name, msgs[2], 0)
            print(f"CONFLICT  (p2 — {p2_name})")
            print(f"  1: {ones}")
            print(f"  0: {zeros}")
            print(f"  match: {json_path.name}\n")
            p2_label = None

        # --- Step 2: player-level fallback (only when tag did not resolve) ---
        if p1_label is None:
            if p1_name.lower() in macro_lower:
                p1_label = 0
            elif p1_name.lower() in cheese_lower:
                p1_label = 1

        if p2_label is None:
            if p2_name.lower() in macro_lower:
                p2_label = 0
            elif p2_name.lower() in cheese_lower:
                p2_label = 1

        # --- No-label warning ---
        if p1_label is None and p2_label is None:
            print(f"WARNING: No labelled strategy found for either player — {json_path.name}")

        game_labels[match_id] = {
            "p1": {"name": p1_name, "label": p1_label},
            "p2": {"name": p2_name, "label": p2_label},
        }

    return game_labels


game_labels = build_game_labels(JSON_DIR, STRATEGIES, MACRO_PLAYERS, CHEESE_PLAYERS)
print(f"\nTotal games processed: {len(game_labels)}")

CONFLICT  (p1 — SharpenedEdge)
  1: ['Tag:build_rush_tempest']
  0: ['Tag:rush_Macro']
  match: match_4190041_metadata.json

CONFLICT  (p2 — sharkbot)
  1: ['Tag:b_workerrush']
  0: ['Tag:b_threegaterobo']
  match: match_4616145_metadata.json

CONFLICT  (p2 — whalemean)
  1: ['Tag:b_dronerush']
  0: ['Tag:b_broodlordbuild']
  match: match_4622275_metadata.json

CONFLICT  (p1 — Nothing)
  1: ['Tag:cheese_12pool']
  0: ['Tag:strat_macro']
  match: match_4731844_metadata.json

CONFLICT  (p1 — Nothing)
  1: ['Tag:cheese_12pool']
  0: ['Tag:strat_macro']
  match: match_4732986_metadata.json

CONFLICT  (p2 — Nothing)
  1: ['Tag:cheese_cannon']
  0: ['Tag:strat_macro']
  match: match_4733570_metadata.json


Total games processed: 651


## Step 2 — Feature Extraction: Bag of Entities

In spam detection a message is a **bag of words** — a count of how often each
word appears.  Here, each player's game is a **bag of entities**:

> **"word"** → SC2 entity type (e.g. `marine`, `zergling`, `gateway`)
> **"word count"** → total units of that type produced during the game

The parquet files track individual entity instances as columns named
`p{1|2}_{botname}_{entity_type}_{seq_id}_{attribute}`.  The highest sequence
ID seen for a given entity type equals the total number of that unit/building
the player ever fielded — e.g. `p2_really_probe_023_*` means 23 probes were
produced.  This is used as the "word frequency."

Each labelled (player, game) pair becomes one training instance.

In [5]:
# Matches per-entity-instance columns:
#   p2_really_probe_023_health  →  player=p2, middle=really_probe, seq=23, attr=health
# The entity type is the last underscore-delimited segment of the "middle" group.
_INST_RE = re.compile(r"^(p[12])_(.+)_(\d{3})_(.+)$")


def extract_entity_counts(parquet_path: Path, player_num: int) -> Counter:
    """
    Build the bag-of-entities for one player in one game.

    Reads only the column schema of the parquet (not row data) and infers
    entity counts from the highest sequence ID seen for each entity type.
    For example, if columns p2_really_probe_001 through p2_really_probe_023
    exist, probe count = 23.  This "total units produced" count is the
    direct analogue of word frequency in a bag-of-words document.

    Note: the parquet also contains aggregate p{n}_{type}_count columns but
    those are unpopulated (always 0) so the per-instance column approach is used.

    Args:
        parquet_path: Absolute path to the *_game_state.parquet file.
        player_num:   1 or 2 — which player's columns to read.

    Returns:
        Counter: {entity_type_name: total_units_produced} — only entries > 0.

    Called by: build_dataset
    """
    import pyarrow.parquet as pq

    # Read only the column schema — avoids loading all row data just for column names.
    schema = pq.read_schema(str(parquet_path))
    player_prefix = f"p{player_num}"
    bag: Counter = Counter()

    for col in schema.names:
        match = _INST_RE.match(col)
        if not match or match.group(1) != player_prefix:
            continue
        middle      = match.group(2)          # e.g. "really_probe"
        seq_id      = int(match.group(3))     # e.g. 23
        entity_type = middle.rsplit("_", 1)[-1]  # e.g. "probe"
        if seq_id > bag[entity_type]:
            bag[entity_type] = seq_id

    return bag


def build_dataset(
    game_labels: Dict[str, dict],
    parquet_dir: Path,
) -> List[Tuple[Counter, int]]:
    """
    Pair every labelled (game, player) with its bag-of-entities representation.

    Skips players whose label is None (unlabelled or conflicted).  Each
    remaining (player, game) pair is one row in the dataset — a direct
    analogue of one labelled email in the spam classifier.

    Args:
        game_labels: Output of build_game_labels().
        parquet_dir: Directory containing *_game_state.parquet files.

    Returns:
        List of (bag_of_entities, label) tuples.
        bag_of_entities: Counter  {entity_name: total_produced}
        label:           1 = cheese, 0 = macro

    Called by: top-level dataset construction code.
    """
    dataset: List[Tuple[Counter, int]] = []

    for match_id, players in game_labels.items():
        parquet_path = parquet_dir / f"{match_id}_game_state.parquet"
        if not parquet_path.exists():
            print(f"WARNING: parquet not found — {parquet_path.name}")
            continue

        for player_key, player_num in [("p1", 1), ("p2", 2)]:
            label = players[player_key]["label"]
            if label is None:
                continue

            bag = extract_entity_counts(parquet_path, player_num)

            # Eris-specific override: the bot author confirmed that under certain
            # defensive conditions, Eris can switch from its macro strategy to a
            # cheesy muta build.  Reclassify any Eris game labelled macro (0) where
            # more than 5 mutalisks were produced as cheese (1) instead.
            player_name = players[player_key]["name"]
            if label == 0 and player_name.lower() == "eris" and bag.get("mutalisk", 0) > 5:
                label = 1
            dataset.append((bag, label))

    return dataset


dataset = build_dataset(game_labels, PARQUET_DIR)

count_cheese = sum(1 for _, lbl in dataset if lbl == 1)
count_macro  = sum(1 for _, lbl in dataset if lbl == 0)

print(f"Number of cheese instances (label 1): {count_cheese}")
print(f"Number of macro instances  (label 0): {count_macro}")
print(f"Total labelled instances:             {len(dataset)}")

Number of cheese instances (label 1): 529
Number of macro instances  (label 0): 149
Total labelled instances:             682


## Step 3 — Train / Test Split

The dataset is shuffled and split into a **training set** (used to fit the
model) and a held-out **test set** (used only for final evaluation).

A **stratified** split keeps the cheese/macro ratio consistent in both halves.

Key decisions:
- **`TEST_FRACTION`** — fraction held out (default 20 %).  With a small dataset
  consider 25–30 % to get a meaningful test set.
- **`RANDOM_SEED`** — set to `None` for a different split each run.

In [6]:
# ---------------------------------------------------------------------------
# Configuration — adjust before running
# ---------------------------------------------------------------------------
TEST_FRACTION = 0.20   # fraction of data held out for testing
RANDOM_SEED   = 42     # fixed seed for reproducibility; None = random each run

# ---------------------------------------------------------------------------
# Stratified shuffle split
# ---------------------------------------------------------------------------
# Splitting cheese and macro separately preserves the class ratio in both sets.
cheese_instances = [(bag, lbl) for bag, lbl in dataset if lbl == 1]
macro_instances  = [(bag, lbl) for bag, lbl in dataset if lbl == 0]

rng = random.Random(RANDOM_SEED)
rng.shuffle(cheese_instances)
rng.shuffle(macro_instances)


def _split_at(items, test_fraction):
    """Return (train, test) where test is the first test_fraction of items."""
    n_test = max(1, int(len(items) * test_fraction))
    return items[n_test:], items[:n_test]


cheese_train, cheese_test = _split_at(cheese_instances, TEST_FRACTION)
macro_train,  macro_test  = _split_at(macro_instances,  TEST_FRACTION)

train_data = cheese_train + macro_train
test_data  = cheese_test  + macro_test

rng.shuffle(train_data)
rng.shuffle(test_data)

print(f"Training set : {len(train_data)} instances  "
      f"({sum(1 for _,l in train_data if l==1)} cheese, "
      f"{sum(1 for _,l in train_data if l==0)} macro)")
print(f"Test set     : {len(test_data)} instances  "
      f"({sum(1 for _,l in test_data if l==1)} cheese, "
      f"{sum(1 for _,l in test_data if l==0)} macro)")

Training set : 544 instances  (424 cheese, 120 macro)
Test set     : 134 instances  (105 cheese, 29 macro)


## Step 4 — Multinomial Naive Bayes: Cheese vs Macro

Structure is identical to the spam classifier in `naive_bayes_spam_classifier.ipynb`.
**Laplace smoothing** (`alpha = 1.0`) prevents zero-probability for entities
that only appear in one class.

During prediction, each entity contributes its log-likelihood multiplied by
its peak count — equivalent to listing it that many times, like a repeated word.

In [7]:
@dataclass
class CheeseClassifier:
    """
    Trained Multinomial Naive Bayes model for SC2 cheese detection.

    Attributes:
        log_prior_cheese:      log P(cheese) from training class counts.
        log_prior_macro:       log P(macro) from training class counts.
        log_likelihood_cheese: {entity: log P(entity | cheese)}
        log_likelihood_macro:  {entity: log P(entity | macro)}
        log_unknown_cheese:    fallback log-likelihood for unseen entities (cheese).
        log_unknown_macro:     fallback log-likelihood for unseen entities (macro).
    """
    log_prior_cheese:      float
    log_prior_macro:       float
    log_likelihood_cheese: Dict[str, float]
    log_likelihood_macro:  Dict[str, float]
    log_unknown_cheese:    float
    log_unknown_macro:     float


def train(
    data: List[Tuple[Counter, int]],
    alpha: float = 1.0,
) -> CheeseClassifier:
    """
    Train the Multinomial Naive Bayes cheese classifier.

    Counts total entity occurrences across all cheese games and all macro games
    separately, then computes Laplace-smoothed log-likelihoods for each entity.

    Args:
        data:  List of (bag_of_entities, label) tuples.  label: 1=cheese, 0=macro.
        alpha: Laplace smoothing factor (default 1.0).

    Returns:
        Trained CheeseClassifier ready for prediction.

    Called by: top-level training code.
    """
    cheese_bags = [bag for bag, lbl in data if lbl == 1]
    macro_bags  = [bag for bag, lbl in data if lbl == 0]

    log_prior_cheese = math.log(len(cheese_bags) / len(data))
    log_prior_macro  = math.log(len(macro_bags)  / len(data))

    # Aggregate entity counts across all games in each class
    cheese_counts: Counter = Counter()
    macro_counts:  Counter = Counter()
    for bag in cheese_bags:
        cheese_counts.update(bag)
    for bag in macro_bags:
        macro_counts.update(bag)

    vocab = set(cheese_counts) | set(macro_counts)
    V = len(vocab)

    cheese_total = sum(cheese_counts.values())
    macro_total  = sum(macro_counts.values())

    denom_cheese = cheese_total + alpha * V
    denom_macro  = macro_total  + alpha * V

    log_likelihood_cheese: Dict[str, float] = {}
    log_likelihood_macro:  Dict[str, float] = {}

    for entity in vocab:
        log_likelihood_cheese[entity] = math.log(
            (cheese_counts[entity] + alpha) / denom_cheese
        )
        log_likelihood_macro[entity] = math.log(
            (macro_counts[entity] + alpha) / denom_macro
        )

    log_unknown_cheese = math.log(alpha / denom_cheese)
    log_unknown_macro  = math.log(alpha / denom_macro)

    return CheeseClassifier(
        log_prior_cheese, log_prior_macro,
        log_likelihood_cheese, log_likelihood_macro,
        log_unknown_cheese, log_unknown_macro,
    )


def predict(model: CheeseClassifier, bag: Counter) -> int:
    """
    Classify a bag-of-entities as cheese (1) or macro (0).

    Each entity contributes its log-likelihood multiplied by its peak count —
    equivalent to listing it that many times, just like a repeated word.

    Args:
        model: Trained CheeseClassifier.
        bag:   Counter {entity_name: peak_count} for the game to classify.

    Returns:
        1 if the model predicts cheese, 0 if macro.

    Called by: evaluate, and for predictions on new games.
    """
    log_cheese = model.log_prior_cheese
    log_macro  = model.log_prior_macro

    for entity, count in bag.items():
        ll_cheese = model.log_likelihood_cheese.get(entity, model.log_unknown_cheese)
        ll_macro  = model.log_likelihood_macro.get(entity,  model.log_unknown_macro)
        log_cheese += count * ll_cheese
        log_macro  += count * ll_macro

    return 1 if log_cheese > log_macro else 0


# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
model = train(train_data)
print("Model trained.")
print(f"  Vocabulary size : {len(model.log_likelihood_cheese)} entity types")
print(f"  log P(cheese)   : {model.log_prior_cheese:.4f}")
print(f"  log P(macro)    : {model.log_prior_macro:.4f}")

Model trained.
  Vocabulary size : 105 entity types
  log P(cheese)   : -0.2492
  log P(macro)    : -1.5115


## Step 5 — Evaluation

Evaluate the trained model on the held-out test set.

- **Accuracy** — fraction of correct predictions overall.
- **Precision (cheese)** — of games predicted cheese, how many actually were?
- **Recall (cheese)** — of actual cheese games, how many did the model catch?
- **F1 (cheese)** — harmonic mean of precision and recall.

With a small test set these numbers are noisy — they establish a baseline
before expanding the dataset.

In [8]:
def evaluate(
    model: CheeseClassifier,
    test_data: List[Tuple[Counter, int]],
) -> None:
    """
    Evaluate the model on held-out test data and print a classification report.

    Prints each per-instance prediction, overall accuracy, precision, recall,
    and F1 for the cheese class, plus a 2x2 confusion matrix.

    Args:
        model:     Trained CheeseClassifier.
        test_data: List of (bag_of_entities, true_label) tuples.

    Returns:
        None — results are printed directly.

    Called by: top-level evaluation code.
    """
    tp = fp = tn = fn = 0

    print("Per-instance predictions:")
    print(f"  {'Predicted':<12} {'Actual':<10} Result")
    print(f"  {'-'*36}")

    for bag, true_label in test_data:
        pred = predict(model, bag)
        pred_str   = "cheese" if pred       == 1 else "macro"
        actual_str = "cheese" if true_label == 1 else "macro"
        correct    = "CORRECT" if pred == true_label else "WRONG"
        print(f"  {pred_str:<12} {actual_str:<10} {correct}")

        if   pred == 1 and true_label == 1: tp += 1
        elif pred == 1 and true_label == 0: fp += 1
        elif pred == 0 and true_label == 0: tn += 1
        else:                               fn += 1

    total     = tp + fp + tn + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp)    if (tp + fp) else 0.0
    recall    = tp / (tp + fn)    if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0.0)

    print()
    print(f"Accuracy  : {accuracy:.2%}  ({tp + tn}/{total} correct)")
    print(f"Precision : {precision:.2%}  (cheese)")
    print(f"Recall    : {recall:.2%}  (cheese)")
    print(f"F1        : {f1:.2%}  (cheese)")
    print()
    precision_macro = tn / (tn + fn)    if (tn + fn) else 0.0
    recall_macro    = tn / (tn + fp)    if (tn + fp) else 0.0
    f1_macro        = (2 * precision_macro * recall_macro / (precision_macro + recall_macro)
                       if (precision_macro + recall_macro) else 0.0)
    print(f"Precision : {precision_macro:.2%}  (macro)")
    print(f"Recall    : {recall_macro:.2%}  (macro)")
    print(f"F1        : {f1_macro:.2%}  (macro)")
    print()
    print("Confusion matrix:")
    print(f"                    Predicted cheese  Predicted macro")
    print(f"  Actual cheese           {tp:>3}              {fn:>3}")
    print(f"  Actual macro            {fp:>3}              {tn:>3}")


evaluate(model, test_data)

Per-instance predictions:
  Predicted    Actual     Result
  ------------------------------------
  macro        cheese     WRONG
  cheese       cheese     CORRECT
  macro        macro      CORRECT
  cheese       macro      WRONG
  macro        macro      CORRECT
  macro        cheese     WRONG
  cheese       cheese     CORRECT
  macro        macro      CORRECT
  macro        cheese     WRONG
  cheese       cheese     CORRECT
  macro        macro      CORRECT
  macro        cheese     WRONG
  cheese       macro      WRONG
  cheese       cheese     CORRECT
  cheese       cheese     CORRECT
  macro        cheese     WRONG
  cheese       cheese     CORRECT
  macro        macro      CORRECT
  macro        cheese     WRONG
  macro        macro      CORRECT
  cheese       cheese     CORRECT
  macro        cheese     WRONG
  macro        macro      CORRECT
  cheese       cheese     CORRECT
  macro        cheese     WRONG
  macro        cheese     WRONG
  macro        cheese     WRONG
  macro 